# Домашнє завдання: Інтерактивні візуалізації з Plotly

## Опис завдання
У цьому домашньому завданні ви будете створювати інтерактивні візуалізації з допомогою бібліотеки Plotly. Ви дізнаєтесь різницю між Plotly Express (швидкі графіки) та Graph Objects (повний контроль), та створите інтерактивний дашборд.

**Опис колонок:**
- `datetime` - дата та час
- `season` - квартал (1-Q1, 2-Q2, 3-Q3, 4-Q4)
- `holiday` - чи є день святковим (0=ні, 1=так)
- `workingday` - чи є день робочим (0=ні, 1=так)
- `weather` - погодні умови (1=ясно, 2=туман, 3=легкий дощ, 4=сильний дощ)
- `temp` - температура в градусах Цельсія
- `atemp` - відчувається як температура
- `humidity` - вологість (%)
- `windspeed` - швидкість вітру
- `casual` - кількість випадкових користувачів
- `registered` - кількість зареєстрованих користувачів
- `count` - загальна кількість орендованих велосипедів

## Підготовка даних

---

🌱 Коментар щодо сезонності

Колонка season у датасеті представляє саме квартали року, а не метеорологічні сезони. Тому всі аналізи сезонності ви можете будувати на основі кварталів.

Водночас дані були зібрані в Індії, де поділ на сезони інший, ніж у Європі чи США. Якщо ви хочете дослідити сезонність відповідно до індійської системи сезонів, можна створити окрему колонку.

Справжні сезони в Індії:

| Сезон        | Місяці                     |
| ------------ | -------------------------- |
| Winter       | December–February (12,1,2) |
| Summer       | March–May (3,4,5)          |
| Monsoon      | June–September (6,7,8,9)   |
| Post-monsoon | October–November (10,11)   |


Тоді потрібно зробити нову колонку weather_season_india, мапнувши місяці так:

12, 1, 2 → 1 (Winter)

3, 4, 5 → 2 (Summer)

6–9 → 3 (Monsoon)

10–11 → 4 (Post-Monsoon)


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# Завантаження даних
from google.colab import drive
drive.mount('/content/drive')
df = pd.read_csv('/content/drive/MyDrive/data/home_tasks/yulu_rental.csv')
df['datetime'] = pd.to_datetime(df['datetime'])

# Для plotly краще не встановлювати datetime як індекс
df['month'] = df['datetime'].dt.month
df['hour'] = df['datetime'].dt.hour
df['weekday'] = df['datetime'].dt.day_name()

# Додаємо назви кварталів в окрему колонку
quarter_map = {1: 'Q1', 2: 'Q2', 3: 'Q3', 4: 'Q4'}
df['quarter_name'] = df['season'].map(quarter_map)

Mounted at /content/drive


## Завдання 1: Базовий інтерактивний лінійний графік (Plotly Express)

**Завдання:**
Створіть інтерактивний лінійний графік динаміки оренди за часом (рівень деталізації - як в даних) з можливістю zoom та hover.

Дайте відповіді на питання.
**Питання для інтерпретації:**
1. Яка перевага інтерактивного графіка над статичним?
2. Чому на графіку є "пробіли" - ділянки, де одна пряма лінія зʼєднує два "суцільних" блоки з даними? Як би ви це могли дослідити на статичному графіку?


In [4]:
fig = px.line(
    df,
    x='datetime',
    y='count',
    title='Динаміка оренди велосипедів у часі',
    labels={
        'datetime': 'Дата',
        'count': 'Оренди'
    }
)

fig.show()

Інтерактивний графік дозволяє збільшувати окремі ділянки та переглядати точні значення. При наведенні курсора на графік можна побачити дату і кількість оренд велосипедів у цю дату. Пробіли виникають тому, що за деякі періоди відсутні записи даних. На статичному графіку це можна було б дослідити за допомогою scatter plot.

## Завдання 2: Scatter plot з додатковими даними (Plotly Express)

**Завдання:**
Створіть scatter plot кількості орендованих велосипедів випадковими користувачами vs кількості орендованих велосипедів зареєстрованими користувачами. Розмір точок встановіть за сумарною кількістю велосипедів, які були взяті в оренду, а колір - за сезоном(кварталом). В hover_data - додайте деталі, які допоможуть вам в подальшому аналізі.

Дослідіть графік. Зверніть увагу, що ви можете вмикати і вимикати окремі квартали, якщо будете клікати на колір кварталу в легенді графіку.

**Дайте відповідь на питання.**
- Як ви проінтерпретуєте роздвоєність цього графіку (дві явні лінії)? Що це означає?
- Які висновки для компанії, які дає велосипеди в оренду, ви можете зробити з цього графіку? 3 основних висновки.

In [7]:
fig = px.scatter(
    df,
    x='casual',
    y='registered',

    size='count',
    color='quarter_name',

    hover_data=[
        'datetime',
        'count',
        'temp',
        'weather',
        'workingday',
        'humidity'
    ],

    title='Кількість орендованих велосипедів випадковими користувачами vs кількості орендованих велосипедів зареєстрованими користувачами',
    labels={
        'casual': 'Випадкові користувачі',
        'registered': 'Зареєстровані користувачі',
        'quarter_name': 'Квартал'
    }
)

fig.show()

Роздвоєність графіку показує, що існують два типи поведінки користувачів. У робочі дні більше оренд роблять зареєстровані користувачі, а у вихідні збільшується кількість випадкових користувачів.
Основні висновки:
1. Зареєстровані користувачі здійснюють більше оренд.
2. Попит зростає одночасно у певний квартал.
3. У вихідні дні зростає попит на оренду велосипедів у випадкових користувачів.

## Завдання 3: Порівняння Plotly Express vs Graph Objects

**Завдання:**
Створіть лінійний графік помісячної динаміки оренди (кількість оренд) велосипедів двома способами - з Plotly Express та з Graph Objects.

**Дайте відповіді на питання.**
1. Як ви розумієте основну різницю між цими двома підходами?
2. Коли краще використовувати Plotly Express?
3. Коли потрібен Graph Objects?


In [9]:
monthly = (
    df.set_index('datetime')
      .resample('M')['count']
      .sum()
      .reset_index()
)

/tmp/ipython-input-159/977395092.py:3: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.



In [10]:
fig_px = px.line(
    monthly,
    x='datetime',
    y='count',
    markers=True,
    title='Помісячна динаміка оренди велосипедів (Plotly Express)',
    labels={'datetime': 'Місяць', 'count': 'Кількість оренд'}
)

fig_px.update_layout(xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
fig_px.show()

In [11]:
fig_go = go.Figure()

fig_go.add_trace(
    go.Scatter(
        x=monthly['datetime'],
        y=monthly['count'],
        mode='lines+markers',
        name='Оренди'
    )
)

fig_go.update_layout(
    title='Помісячна динаміка оренди велосипедів (Graph Objects)',
    xaxis_title='Місяць',
    yaxis_title='Кількість оренд'
)

fig_go.update_xaxes(showgrid=True)
fig_go.update_yaxes(showgrid=True)

fig_go.show()

 Основна різниця між Plotly Express та Graph Objects у тому , що Plotly Express має швидкий та високорівневий інтерфейс, в той час як Graph Objects має низькорівневий інтерфейс.Plotly Express краще використовувати коли потрібно швидко побудувати графіки. Graph Objects потрібен для побудови складних графіків.

## Завдання 4 (Опціональне): Дашборд з make_subplots (Graph Objects)

**Завдання:**
Створіть дашборд з 4 різними графіками в одній фігурі:
- Bar chart - середні значення загальної кількості оренд велосипедів за сезонами(кварталами)
- Pie chart - відсоткове співвідношення погодних умов в даних
- Line chart - середнє значення загальної кількості оренд велосипедів за годинами протягом доби
- Scatter plot - кореляція температури vs вологість

Додайте заголовок на дашборд.

**Дайте відповідь на питання**
- На ваш погляд, яка перевага об'єднання графіків в один дашборд?

In [12]:
season_avg = df.groupby('quarter_name')['count'].mean().reset_index()
weather_counts = df['weather'].value_counts().sort_index()
weather_labels = {
    1: 'Ясно',
    2: 'Туман',
    3: 'Легкий дощ',
    4: 'Сильний дощ'
}
hour_avg = df.groupby('hour')['count'].mean().reset_index()
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "Середня кількість оренд за кварталами",
        "Розподіл погодних умов",
        "Середня кількість оренд за годинами",
        "Температура vs Вологість"
    ),
    specs=[
        [{"type": "bar"}, {"type": "pie"}],
        [{"type": "scatter"}, {"type": "scatter"}]
    ]
)

#1

fig.add_trace(
    go.Bar(
        x=season_avg['quarter_name'],
        y=season_avg['count'],
        marker_color='skyblue',
        name='Оренди'
    ),
    row=1, col=1
)

# 2

fig.add_trace(
    go.Pie(
        labels=[weather_labels[i] for i in weather_counts.index],
        values=weather_counts.values,
        name="Погода"
    ),
    row=1, col=2
)

#3

fig.add_trace(
    go.Scatter(
        x=hour_avg['hour'],
        y=hour_avg['count'],
        mode='lines+markers',
        line=dict(color='red'),
        name='Оренди'
    ),
    row=2, col=1
)

#4

fig.add_trace(
    go.Scatter(
        x=df['temp'],
        y=df['humidity'],
        mode='markers',
        marker=dict(color='purple', opacity=0.5),
        name='Temp vs Humidity'
    ),
    row=2, col=2
)



fig.update_layout(

    title="📊 Dashboard аналізу оренди велосипедів",

    height=800,

    showlegend=False
)

fig.show()

Перевагою вважаю те, що це зручно, одразу можна бачити закономірності, можна побачити повну картину по орендам велосипедів, побачити в які години користуються попитом оренда велосипедів. Виходячи із цих графіків, які одночано розташовані можна вже зробити якісь висновки і приймати певні рішення.



 ## Завдання 5 (Опціональне): 3D візуалізація

**Завдання:**
Створіть 3D scatter plot для аналізу взаємозв'язку температури, швидкості вітру та загальної кількості орендованих велосипедів. Колір встановіть за сезоном(кварталом), а розмір - за загальною кількість оренд також.

Дайте відповіді на питання.
**Питання для інтерпретації:**
1. Яку додаткову інформацію, на ваш погляд, дає 3D візуалізація?
2. Чи видно кластери в 3D просторі?
3. Чи ви можете зробити висновки з цієї візуалізації, чи вам було простіше побудувати кілька 2D?



In [13]:
fig = px.scatter_3d(

    df,

    x='temp',
    y='windspeed',
    z='count',

    color='quarter_name',

    size='count',

    hover_data=[
        'humidity',
        'hour',
        'weekday'
    ],

    title='3D аналіз: Температура, вітер та кількість оренд',

    color_discrete_sequence=px.colors.qualitative.Set2
)

fig.update_layout(

    scene=dict(

        xaxis_title='Температура',

        yaxis_title='Швидкість вітру',

        zaxis_title='Кількість оренд'
    ),

    height=700
)

fig.show()

3D візуалізація дозволяє одночасно побачити вплив декількох факторів  на кількість оренд, а на 2D графіку ми бачимо тільки одну залежність.
Так, видно кластери в 3D просторі. Для детального аналізу мені зручніше використовувати кілька 2D графіків, тому що вони простіші для читання.


## Завдання 6: Експорт та збереження інтерактивних графіків

**Завдання:**
Збережіть побудований попередній графік на plotly в формат HTML. Також змініть вручну щось на графіку (зум, виділення частини графіку) і збережіть його як статичне зображення через іконку фотоапарату у формат PNG. Завантажте файли з графіком у HTML та PNG (або посилання на них на github) разом з посиланням на цей ноутбук при здачі ДЗ.


In [15]:
from google.colab import files
files.download("yulu_3d_scatter.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>